# FlowGuard-MPC: Adaptive Model-Predictive Production Choke Controller for Autonomous Safe Oil Well Optimization
### Honeywell Hackathon Round 2 Deliverable
**Author**: Shyambaskar Sriram (SASTRA University)

This notebook contains the end-to-end implementation of **FlowGuard-MPC**, an Autonomous Model Predictive Controller (MPC) with an explicit Hard-Guard Safety Filter and Candidate Rejection Engine for a single naturally flowing oil well.

## 1. Import Libraries and Resolve Dataset Path
Uses dynamic relative path resolution so the notebook executes seamlessly on any machine or evaluator directory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from system_id import find_dataset_path, DynamicSystemModel

# Resolve dataset path dynamically
dataset_path = find_dataset_path()
print('Resolved Dataset Path:', dataset_path)

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print('Dataset Shape:', df.shape)
    display(df.head(10))
else:
    print('Using dynamic online simulation mode.')

## 2. Dynamic System Identification & Step-Test Analysis
Analyzing the open-loop step test responses to determine process gains for Q, WHP, FLP, and BHP.

In [ ]:
sys_model = DynamicSystemModel(dataset_path=dataset_path)
print('Safety Envelope Limits:', sys_model.safety_envelope)

if sys_model.df is not None:
    # Plot step-test dataset
    fig, axes = plt.subplots(5, 1, figsize=(10, 10), sharex=True)
    axes[0].plot(sys_model.df['Time_hr'], sys_model.df['OilRate_bbl_hr'], 'b-')
    axes[0].set_ylabel('Oil Rate (bbl/hr)')
    axes[1].plot(sys_model.df['Time_hr'], sys_model.df['WHP_psi'], 'g-')
    axes[1].set_ylabel('WHP (psi)')
    axes[2].plot(sys_model.df['Time_hr'], sys_model.df['FLP_psi'], 'm-')
    axes[2].set_ylabel('FLP (psi)')
    axes[3].plot(sys_model.df['Time_hr'], sys_model.df['BHP_psi'], 'c-')
    axes[3].set_ylabel('BHP (psi)')
    axes[4].plot(sys_model.df['Time_hr'], sys_model.df['Choke_pct'], 'k-')
    axes[4].set_ylabel('Choke (%)')
    axes[4].set_xlabel('Time (hr)')
    plt.suptitle('Open-Loop Step Test Telemetry')
    plt.tight_layout()
    plt.show()

## 3. FlowGuard-MPC Autonomous Controller Initialization

In [ ]:
from mpc_controller import AutonomousChokeController
from simulator import OilWellSimulator

sim = OilWellSimulator()
controller = AutonomousChokeController(whp_min=210.0, flp_min=150.0, bhp_min=2850.0, max_ramp=5.0)
print('FlowGuard-MPC initialized with Hard-Guard Safety Filter.')

## 4. Scenario Evaluation & Demonstration Plots
Executing Scenario A (Startup), Scenario B (Target Tracking), and Scenario C (Infeasible Target).

In [ ]:
import importlib
import run_scenarios
importlib.reload(run_scenarios)
from run_scenarios import run_simulation, plot_scenario_results

# Scenario A
df_A, rejections_A = run_simulation('Scenario A', [110.0]*50, timesteps=50, initial_choke=30.0)
print('Scenario A Final Q:', round(df_A['Actual_Q'].iloc[-1], 2), 'bbl/hr')

# Scenario B
df_B, rejections_B = run_simulation('Scenario B', [100.0]*25 + [150.0]*25, timesteps=50, initial_choke=35.0)
print('Scenario B Final Q:', round(df_B['Actual_Q'].iloc[-1], 2), 'bbl/hr')

# Scenario C
df_C, rejections_C = run_simulation('Scenario C', [220.0]*50, timesteps=50, initial_choke=30.0)
print('Scenario C Settled Q:', round(df_C['Actual_Q'].iloc[-1], 2), 'bbl/hr (Safe limit maintained)')
print('Scenario C Candidate Moves Rejected:', len(rejections_C))